# 01 - Data and return diagnostics
Goal: load SGX prices, check data quality, and show the return features (fat tails, volatility clustering) that justify the VaR model choices in later steps.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from src.data import download_prices, log_returns, data_quality_report

prices = download_prices()
rets = log_returns(prices)
prices.tail()

## Data quality checks

In [ ]:
data_quality_report(prices)

## Price history (rebased to 100)

In [ ]:
(prices / prices.bfill().iloc[0] * 100).plot(figsize=(10, 4), title="Rebased prices")
plt.show()

## Stylised facts: tails and non-normality

In [ ]:
summary = pd.DataFrame({
    "ann_vol": rets.std() * np.sqrt(252),
    "skew": rets.skew(),
    "excess_kurtosis": rets.kurt(),
    "jarque_bera_p": rets.apply(lambda s: stats.jarque_bera(s)[1]),
})
summary.round(4)

## Volatility clustering: autocorrelation of squared returns
Significant autocorrelation in squared returns supports using GARCH in Step 3.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, name in zip(axes, ["DBS", "OCBC", "UOB"]):
    plot_acf(rets[name] ** 2, lags=30, ax=ax, title=f"ACF of squared returns: {name}")
plt.tight_layout(); plt.show()

## Observations
*Write 3-4 sentences in your own words here: which names have the fattest tails, what happened in March 2020, and what this means for a normal-distribution VaR.*